# 🪢 Langfuse: наблюдаемость и оценка LLM-приложений

**Практика · ~50 минут**

Возьмём небольшого LLM-агента — «финансовый ассистент» на GigaChat — и научимся:

1. **Трейсинг** — видеть, что происходит внутри агента: шаги, промпты, токены, латентность.
2. **Оценка (Scores)** — измерять качество ответов вручную и программно.
3. **Датасеты и эксперименты** — гонять агента на наборе тест-кейсов и сравнивать версии промптов/моделей.

Это и есть рабочий цикл улучшения LLM-систем:
*наблюдаю → нахожу плохие кейсы → собираю датасет → меняю промпт/модель → сравниваю по метрикам.*

> Стек: **Langfuse v4** (Cloud) + **GigaChat** + **LangChain**.

**План:**
- Шаг 0 — ключи и подключение (5 мин)
- Часть 1 — первый трейс с `@observe` (5 мин)
- Часть 2 — ручная инструментация: спаны и генерации (12 мин)
- Часть 3 — автотрейсинг через LangChain callback (8 мин)
- Часть 4 — оценка (Scores) (8 мин)
- Часть 5 — датасеты и эксперименты (10 мин)
- ⭐ Бонусы и итоги

## Шаг 0. Регистрация и ключи Langfuse

1. Откройте **https://cloud.langfuse.com** и зарегистрируйтесь (можно через Google/GitHub).
2. Создайте **организацию** и **проект** (например, `hse-seminar`).
3. **Settings → API Keys → Create new API keys**.
4. Скопируйте **Public key** (`pk-lf-...`) и **Secret key** (`sk-lf-...`). Secret показывается один раз.

Регион по умолчанию — **EU** (`https://cloud.langfuse.com`). Для US — `https://us.cloud.langfuse.com`.

> Если ключи лежат в `.env` (как в этом проекте) — они подхватятся автоматически.
> Иначе ноутбук спросит их через `getpass`, чтобы они **не сохранялись** в файл.

In [ ]:
# %pip install -q "langfuse>=4.0" langchain-gigachat python-dotenv

In [ ]:
import os, getpass
from dotenv import load_dotenv, find_dotenv

# Креды GigaChat и (если есть) ключи Langfuse берём из .env проекта.
# usecwd=True — ищем .env от текущей папки вверх (надёжно и в Jupyter, и при headless-запуске).
load_dotenv(find_dotenv(usecwd=True), override=True)

# Если ключей Langfuse в окружении нет — спросим вручную (не попадут в сохранённый ноутбук)
def _ask(var, prompt):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(prompt)

_ask("LANGFUSE_PUBLIC_KEY", "Langfuse Public Key (pk-lf-...): ")
_ask("LANGFUSE_SECRET_KEY", "Langfuse Secret Key (sk-lf-...): ")
# EU по умолчанию; для US замените на https://us.cloud.langfuse.com
os.environ.setdefault("LANGFUSE_BASE_URL", "https://cloud.langfuse.com")
print("ENV готов. Хост:", os.environ["LANGFUSE_BASE_URL"])

In [ ]:
from langfuse import Langfuse, get_client, observe, propagate_attributes, Evaluation

langfuse = Langfuse(
    public_key=os.environ["LANGFUSE_PUBLIC_KEY"],
    secret_key=os.environ["LANGFUSE_SECRET_KEY"],
    base_url=os.environ["LANGFUSE_BASE_URL"],
)

assert langfuse.auth_check(), "Auth не прошёл — проверьте ключи и регион (EU/US)"
print("✅ Langfuse подключён:", os.environ["LANGFUSE_BASE_URL"])

In [ ]:
from langchain_gigachat import GigaChat
from langchain_core.messages import SystemMessage, HumanMessage

# Креды подхватываются из переменных окружения (GIGACHAT_CREDENTIALS и т.д.)
llm = GigaChat(verify_ssl_certs=False, model="GigaChat-2", temperature=0.1, timeout=45)

print(llm.invoke([HumanMessage(content="Ответь одним словом: работает?")]).content)

## Часть 1. Первый трейс с `@observe`

Базовые понятия Langfuse:
- **Trace (трейс)** — один полный запрос к системе (например, один вопрос пользователя).
- **Observation (наблюдение)** внутри трейса бывает трёх типов:
  - **span** — произвольный шаг (поиск по базе, парсинг, вызов инструмента);
  - **generation** — вызов LLM (промпт, ответ, токены, стоимость);
  - **event** — точечное событие.

Самый быстрый способ — декоратор `@observe()`: он сам создаёт трейс вокруг функции
и записывает её **входы, выходы, время и ошибки**.

In [ ]:
@observe(name="summarize")
def summarize(text: str) -> str:
    msg = [
        SystemMessage(content="Ты кратко пересказываешь текст одним предложением на русском."),
        HumanMessage(content=text),
    ]
    return llm.invoke(msg).content

out = summarize("Langfuse — это опенсорс-платформа для наблюдаемости и оценки LLM-приложений.")
print(out)

get_client().flush()  # в ноутбуках/скриптах не забывайте flush, иначе данные не уйдут на сервер
print("→ Откройте проект в Langfuse → Tracing и найдите трейс 'summarize'")

### 🛠 Задание 1
Задекорируйте свою функцию через `@observe()` (например, перевод текста или генерация заголовка),
вызовите её и найдите трейс в UI. Обратите внимание на input/output и на вложенную **generation** от GigaChat.

In [ ]:
@observe(name="generate-title")
def generate_title(text: str) -> str:
    """Генерирует короткий заголовок для переданного текста."""
    messages = [
        SystemMessage(
            content=(
                "Ты редактор учебных материалов. Придумай один короткий, "
                "понятный заголовок на русском языке. Верни только заголовок."
            )
        ),
        HumanMessage(content=text),
    ]
    return llm.invoke(messages).content.strip()

sample_text = (
    "Langfuse помогает разработчикам LLM-приложений видеть трейсинг, "
    "латентность, стоимость и качество ответов модели."
)
print(generate_title(sample_text))

get_client().flush()
print("→ Трейс 'generate-title' отправлен в Langfuse")


## Часть 2. Ручная инструментация: спаны и генерации

Декоратор удобен, но часто нужно видеть **внутреннюю структуру** шага. Соберём мини-RAG:
«финансовый ассистент» отвечает на вопросы о (вымышленных) компаниях, опираясь на базу знаний.

Структура трейса, которую построим:
```
financial-assistant            (trace)
├── retrieve                   (span)        — поиск релевантных документов
└── giga-answer                (generation)  — ответ GigaChat по контексту
```
Спаны/генерации создаём контекст-менеджером `start_as_current_observation(as_type=...)` —
они автоматически вкладываются друг в друга.

In [ ]:
KB = [
    {"company": "НоваЭнерджи",    "text": "НоваЭнерджи — компания зелёной энергетики: солнечные и ветровые станции. Выручка растёт ~30% в год, но высокий долг."},
    {"company": "НексаКор",       "text": "НексаКор — облачная инфраструктура и ИИ-чипы. Лидер рынка, высокая маржа, но дорогая оценка акций."},
    {"company": "ПетроГлобал",    "text": "ПетроГлобал — нефтегазовый холдинг. Стабильные дивиденды, но прибыль сильно зависит от цен на нефть и регуляторных рисков."},
    {"company": "УрбанМарт",      "text": "УрбанМарт — сеть розничного ритейла. Тонкая маржа, экспансия в регионы, жёсткая конкуренция с маркетплейсами."},
    {"company": "ВелоситиМоторс", "text": "ВелоситиМоторс — производитель электромобилей. Быстрый рост поставок, но пока убыточна из-за проблем с цепочкой поставок."},
]

def naive_retrieve(query: str, k: int = 2):
    # Простейший «поиск» по пересечению слов запроса и документа (без эмбеддингов — для наглядности).
    q = set(query.lower().replace("?", " ").split())
    scored = []
    for doc in KB:
        words = set((doc["company"] + " " + doc["text"]).lower().split())
        scored.append((len(q & words), doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scored[:k]]

print([d["company"] for d in naive_retrieve("Чем занимается НоваЭнерджи?")])

In [ ]:
SYSTEM_PROMPT = (
    "Ты — финансовый ассистент. Отвечай кратко (2-3 предложения) на русском "
    "и опирайся ТОЛЬКО на предоставленный контекст. Если данных нет — честно скажи об этом."
)

@observe(name="financial-assistant")
def answer_question(question: str) -> str:
    client = get_client()

    # ШАГ 1 — поиск (span)
    with client.start_as_current_observation(as_type="span", name="retrieve") as span:
        docs = naive_retrieve(question, k=2)
        span.update(
            input={"question": question},
            output={"found": [d["company"] for d in docs]},
        )

    context = "\n".join(f"- {d['text']}" for d in docs)
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"Контекст:\n{context}\n\nВопрос: {question}"),
    ]

    # ШАГ 2 — ответ LLM (generation)
    with client.start_as_current_observation(
        as_type="generation", name="giga-answer", model="GigaChat-2",
    ) as gen:
        ai = llm.invoke(messages)
        usage = ai.usage_metadata or {}
        gen.update(
            input=[m.content for m in messages],
            output=ai.content,
            usage_details={
                "input": usage.get("input_tokens", 0),
                "output": usage.get("output_tokens", 0),
                "total": usage.get("total_tokens", 0),
            },
        )
    return ai.content


# Вызов с трейс-атрибутами: пользователь, сессия, теги (применятся ко всему трейсу)
with propagate_attributes(user_id="student-01", session_id="seminar-demo", tags=["manual", "rag"]):
    print(answer_question("Стоит ли обратить внимание на НоваЭнерджи?"))

get_client().flush()
print("→ В трейсе 'financial-assistant' видны span 'retrieve' и generation 'giga-answer' с токенами")

В UI откройте трейс **financial-assistant**: дерево из `retrieve` и `giga-answer`,
входы/выходы каждого шага, число токенов и латентность, а также теги и user/session.

### 🛠 Задание 2
Добавьте в агента **инструмент** — функцию `get_price(company)` (ниже) — как отдельный **span** `get-price`,
и включите цену в контекст ответа.

In [ ]:
PRICES = {
    "НоваЭнерджи": 124.5,
    "НексаКор": 980.0,
    "ПетроГлобал": 311.2,
    "УрбанМарт": 56.7,
    "ВелоситиМоторс": 88.3,
}

def get_price(company: str):
    """Возвращает текущую цену акции компании из учебного словаря."""
    return PRICES.get(company)


@observe(name="financial-assistant")
def answer_question(question: str) -> str:
    """Финансовый ассистент с ручной инструментрацией retrieve/get-price/generation."""
    client = get_client()

    # ШАГ 1 — поиск релевантных документов.
    with client.start_as_current_observation(as_type="span", name="retrieve") as span:
        docs = naive_retrieve(question, k=2)
        span.update(
            input={"question": question},
            output={"found": [d["company"] for d in docs]},
        )

    # ШАГ 2 — инструмент получения цены по первой найденной компании.
    main_company = docs[0]["company"] if docs else None
    with client.start_as_current_observation(as_type="span", name="get-price") as span:
        price = get_price(main_company) if main_company else None
        span.update(
            input={"company": main_company},
            output={"price": price},
        )

    context_lines = [f"- {d['text']}" for d in docs]
    if main_company is not None:
        price_text = "нет данных" if price is None else f"{price}"
        context_lines.append(f"- Текущая цена акции {main_company}: {price_text}.")

    context = "\n".join(context_lines)
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"Контекст:\n{context}\n\nВопрос: {question}"),
    ]

    # ШАГ 3 — ответ LLM.
    with client.start_as_current_observation(
        as_type="generation",
        name="giga-answer",
        model="GigaChat-2",
    ) as gen:
        ai = llm.invoke(messages)
        usage = ai.usage_metadata or {}
        gen.update(
            input=[m.content for m in messages],
            output=ai.content,
            usage_details={
                "input": usage.get("input_tokens", 0),
                "output": usage.get("output_tokens", 0),
                "total": usage.get("total_tokens", 0),
            },
        )
    return ai.content


with propagate_attributes(user_id="student-01", session_id="seminar-demo", tags=["manual", "rag", "tool"]):
    print(answer_question("Стоит ли обратить внимание на НоваЭнерджи и какая у неё цена?"))

get_client().flush()
print("→ В трейсе 'financial-assistant' теперь есть span 'get-price'")


## Часть 3. Автоматическая инструментация через LangChain

Если вы используете LangChain/LangGraph, ручные спаны не нужны — есть `CallbackHandler`,
который сам трейсит все шаги цепочки/агента.

```python
from langfuse.langchain import CallbackHandler
handler = CallbackHandler()
chain.invoke(x, config={"callbacks": [handler]})
```
Трейс-атрибуты передаются через `metadata` с ключами
`langfuse_session_id`, `langfuse_user_id`, `langfuse_tags`.

In [ ]:
from langfuse.langchain import CallbackHandler
from langchain_core.prompts import ChatPromptTemplate

handler = CallbackHandler()

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Контекст:\n{context}\n\nВопрос: {question}"),
])
chain = prompt | llm

question = "Какой главный риск у ПетроГлобал?"
context = "\n".join(f"- {d['text']}" for d in naive_retrieve(question))

resp = chain.invoke(
    {"context": context, "question": question},
    config={
        "callbacks": [handler],
        "metadata": {
            "langfuse_session_id": "seminar-demo",
            "langfuse_user_id": "student-01",
            "langfuse_tags": ["auto", "langchain"],
        },
    },
)
print(resp.content)
get_client().flush()
print("→ Трейс создан автоматически; токены GigaChat подтянулись из ответа")

Сравните трейс из Части 2 (ручной) и этот (авто): содержимое похоже, но во втором случае
мы не писали ни одного спана вручную.

### 🛠 Задание 3
Прогоните цепочку на 3 разных вопросах с **одинаковым** `langfuse_session_id`.
В разделе **Sessions** они сгруппируются в один диалог.

In [ ]:
questions = [
    "Чем занимается НоваЭнерджи?",
    "Какой главный риск у ПетроГлобал?",
    "Что выпускает ВелоситиМоторс?",
]

session_id = "dialog-1"

for q in questions:
    ctx = "\n".join(f"- {d['text']}" for d in naive_retrieve(q))
    response = chain.invoke(
        {"context": ctx, "question": q},
        config={
            "callbacks": [CallbackHandler()],
            "metadata": {
                "langfuse_session_id": session_id,
                "langfuse_user_id": "student-01",
                "langfuse_tags": ["auto", "langchain", "session-demo"],
            },
        },
    )
    print(f"Q: {q}\nA: {response.content}\n")

get_client().flush()
print(f"→ 3 запроса отправлены в одну сессию: {session_id}")


## Часть 4. Оценка (Scores)

**Score** — числовая/категориальная/булева оценка трейса или наблюдения. Источники:
- **вручную** в UI (аннотации — удобно для разметки «хорошо/плохо»);
- **программно** из кода (правила, эвристики);
- **LLM-as-judge** (модель оценивает ответ — см. бонус).

Типы: `NUMERIC` (0.85), `CATEGORICAL` ("relevant"), `BOOLEAN` (0/1).

Запишем программную оценку прямо внутри трейса агента.

In [ ]:
@observe(name="financial-assistant-scored")
def answer_and_score(question: str) -> str:
    client = get_client()
    docs = naive_retrieve(question, k=2)
    context = "\n".join(f"- {d['text']}" for d in docs)
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"Контекст:\n{context}\n\nВопрос: {question}"),
    ]
    answer = llm.invoke(messages).content

    # Эвристика «опоры на контекст»: упомянута ли найденная компания в ответе
    grounded = any(d["company"] in answer for d in docs)
    client.score_current_trace(
        name="grounded",
        value=1 if grounded else 0,
        data_type="BOOLEAN",
        comment="Ответ упоминает найденную компанию" if grounded else "Компания не упомянута",
    )
    # Длина ответа как числовая метрика
    client.score_current_trace(name="answer_length", value=len(answer), data_type="NUMERIC")
    return answer

print(answer_and_score("Что выпускает ВелоситиМоторс?"))
get_client().flush()
print("→ В трейсе появились скоры 'grounded' и 'answer_length'")

В UI трейс можно оценить и **вручную**: откройте трейс → **Add score** (например, `useful` = 👍/👎).
Это базовый способ собрать разметку для последующих экспериментов.

### 🛠 Задание 4
Добавьте свой score: например, `refusal` (BOOLEAN) — отказался ли ассистент отвечать
(ищите в ответе «не знаю» / «нет данных»).

In [ ]:
@observe(name="financial-assistant-scored")
def answer_and_score(question: str) -> str:
    client = get_client()
    docs = naive_retrieve(question, k=2)
    context = "\n".join(f"- {d['text']}" for d in docs)
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"Контекст:\n{context}\n\nВопрос: {question}"),
    ]
    answer = llm.invoke(messages).content

    # Эвристика «опоры на контекст»: упомянута ли найденная компания в ответе.
    grounded = any(d["company"] in answer for d in docs)
    client.score_current_trace(
        name="grounded",
        value=1 if grounded else 0,
        data_type="BOOLEAN",
        comment="Ответ упоминает найденную компанию" if grounded else "Компания не упомянута",
    )

    # Длина ответа как числовая метрика.
    client.score_current_trace(
        name="answer_length",
        value=len(answer),
        data_type="NUMERIC",
        comment="Длина ответа, символов",
    )

    # Новый score: отказался ли ассистент отвечать из-за нехватки данных.
    refusal_markers = [
        "не знаю",
        "нет данных",
        "недостаточно данных",
        "не могу ответить",
        "не могу дать",
        "данных нет",
    ]
    refusal = any(marker in answer.lower() for marker in refusal_markers)
    client.score_current_trace(
        name="refusal",
        value=1 if refusal else 0,
        data_type="BOOLEAN",
        comment="Ассистент отказался отвечать" if refusal else "Ассистент дал содержательный ответ",
    )

    return answer


print(answer_and_score("Что выпускает ВелоситиМоторс?"))
print(answer_and_score("Какая дивидендная политика у компании КосмоФуд?"))
get_client().flush()
print("→ В трейсе появились скоры 'grounded', 'answer_length' и 'refusal'")


## Часть 5. Датасеты и эксперименты

Чтобы не проверять агента «на глаз», собирают **датасет** тест-кейсов (вход + эталон) и запускают
**эксперимент**: прогнать агента по всем кейсам и оценить каждый ответ. Разные запуски
(промпты/модели) сравниваются по метрикам бок о бок.

Поток: `create_dataset → create_dataset_item → get_dataset → dataset.run_experiment(task, evaluators)`.

In [ ]:
DATASET_NAME = "companies-qa"

# create_dataset идемпотентен по имени — повторный запуск не создаёт дубль датасета
langfuse.create_dataset(name=DATASET_NAME, description="Вопросы о компаниях для семинара")

ITEMS = [
    {"input": "Чем занимается НоваЭнерджи?",            "expected_output": "энерг"},
    {"input": "Какой главный риск у ПетроГлобал?",      "expected_output": "нефт"},
    {"input": "Что выпускает ВелоситиМоторс?",          "expected_output": "электромоб"},
    {"input": "В каком секторе работает УрбанМарт?",    "expected_output": "ритейл"},
    {"input": "Почему акции НексаКор считаются дорогими?", "expected_output": "оценк"},
]

for i, it in enumerate(ITEMS):
    langfuse.create_dataset_item(
        dataset_name=DATASET_NAME,
        id=f"q{i}",  # стабильный id → повторный запуск обновляет, а не плодит дубли
        input=it["input"],
        expected_output=it["expected_output"],
    )
print(f"Датасет '{DATASET_NAME}': {len(ITEMS)} кейсов")

In [ ]:
from datetime import datetime

def task(*, item, **kwargs):
    # item из датасета Langfuse → доступ через атрибут .input
    question = item.input if hasattr(item, "input") else item["input"]
    return answer_question(question)  # переиспользуем инструментированного агента из Части 2

def contains_expected(*, input, output, expected_output=None, **kwargs):
    ok = bool(expected_output) and expected_output.lower() in (output or "").lower()
    return Evaluation(
        name="contains_expected",
        value=1.0 if ok else 0.0,
        comment=f"Ожидали подстроку '{expected_output}'",
    )

def answer_length(*, output, **kwargs):
    return Evaluation(name="answer_length", value=len(output or ""), comment="Длина ответа, символов")

dataset = langfuse.get_dataset(DATASET_NAME)
result = dataset.run_experiment(
    name=f"baseline-{datetime.now():%H%M%S}",
    description="GigaChat-2, базовый промпт",
    task=task,
    evaluators=[contains_expected, answer_length],
)
print(result.format())
get_client().flush()

В UI: **Datasets → companies-qa → Runs**. Виден прогон со средними метриками. Запустив второй прогон
(другой промпт/модель), вы **сравните** их бок о бок — это и есть основа оптимизации промптов
(подход *auto-prompting*).

### 🛠 Задание 5
Измените `SYSTEM_PROMPT` (или `temperature`/модель), запустите второй эксперимент с другим `name`
и сравните метрику `contains_expected` двух прогонов в UI.

In [ ]:
SYSTEM_PROMPT_V2 = (
    "Ты — осторожный финансовый ассистент. Отвечай кратко на русском языке. "
    "Сначала назови компанию, затем главный факт и главный риск. "
    "Опирайся ТОЛЬКО на предоставленный контекст. "
    "Если в контексте нет ответа, прямо скажи: 'нет данных'."
)


def answer_question_v2(question: str) -> str:
    """Вариант агента для второго эксперимента: более строгий и структурированный промпт."""
    docs = naive_retrieve(question, k=2)
    context = "\n".join(f"- {d['text']}" for d in docs)
    messages = [
        SystemMessage(content=SYSTEM_PROMPT_V2),
        HumanMessage(content=f"Контекст:\n{context}\n\nВопрос: {question}"),
    ]
    return llm.invoke(messages).content


def task_v2(*, item, **kwargs):
    question = item.input if hasattr(item, "input") else item["input"]
    return answer_question_v2(question)


dataset = langfuse.get_dataset(DATASET_NAME)
result_v2 = dataset.run_experiment(
    name=f"variant-strict-{datetime.now():%H%M%S}",
    description="GigaChat-2, более строгий промпт: компания → факт → риск; отказ при нехватке данных",
    task=task_v2,
    evaluators=[contains_expected, answer_length],
)
print(result_v2.format())
get_client().flush()
print("→ Второй эксперимент создан; сравните baseline-* и variant-strict-* в Langfuse UI")


## ⭐ Бонус. LLM-as-judge

Эталонная подстрока — грубая метрика. Точнее — оценивать ответ другой моделью.
Добавим evaluator, который просит GigaChat оценить релевантность ответа вопросу по шкале 0–1.

In [ ]:
import re

def llm_judge_relevance(*, input, output, expected_output=None, **kwargs):
    q = input if isinstance(input, str) else str(input)
    judge_msg = [
        SystemMessage(content="Ты — строгий оценщик. Верни ТОЛЬКО число от 0 до 1 — насколько ответ релевантен вопросу."),
        HumanMessage(content=f"Вопрос: {q}\nОтвет: {output}\nОценка (0..1):"),
    ]
    raw = llm.invoke(judge_msg).content
    m = re.search(r"[01](?:[.,]\d+)?", raw)
    score = float(m.group().replace(",", ".")) if m else 0.0
    return Evaluation(
        name="llm_relevance",
        value=min(max(score, 0.0), 1.0),
        comment=f"Вердикт модели: {raw.strip()[:80]}",
    )

dataset = langfuse.get_dataset(DATASET_NAME)
result = dataset.run_experiment(
    name=f"with-judge-{datetime.now():%H%M%S}",
    description="Добавлен LLM-as-judge",
    task=task,
    evaluators=[contains_expected, answer_length, llm_judge_relevance],
)
print(result.format())
get_client().flush()

## Итоги

Вы прошли полный цикл наблюдаемости в Langfuse:
- **Трейсинг** — `@observe`, ручные `span`/`generation`, авто-трейсинг через LangChain `CallbackHandler`;
- **Контекст** — сессии, пользователи, теги, токены/латентность;
- **Оценка** — программные scores и LLM-as-judge;
- **Датасеты и эксперименты** — регрессионное сравнение версий агента.

**Рабочий цикл:** прод-трейсы → находим плохие кейсы → добавляем в датасет → меняем промпт/модель →
эксперимент → сравниваем метрики → катим лучшее.

### Полезные ссылки
- Документация: https://langfuse.com/docs
- Python SDK: https://langfuse.com/docs/observability/sdk/python/overview
- Эксперименты: https://langfuse.com/docs/evaluation/experiments/experiments-via-sdk
- LangChain-интеграция: https://langfuse.com/docs/integrations/langchain/tracing

```python
# В конце скрипта/сервиса всегда сбрасывайте буфер:
get_client().flush()       # отправить накопленные события
# get_client().shutdown()  # при завершении процесса
```